# LID Research — Week 2: Baseline Reproduction
**Goal:** Reproduce DoLA, LSD, SSP on TruthfulQA-dev with Llama-3-8B

**KPIs this notebook must hit:**
| Baseline | Target AUROC | Pass Condition |
|---|---|---|
| DoLA | ≥ 0.60 | Within ±2pp of Chuang et al. |
| LSD  | ≥ 0.58 | Within ±2pp of paper |
| SSP  | ≥ 0.58 | Within ±3pp of Luo 2025 |

**These numbers become the bar LID must beat in Week 5.**

---
⚡ Runtime: A100 GPU required (university cluster or Colab Pro+)
🕐 Estimated time: ~45 minutes total

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
import subprocess, sys

print('Installing dependencies...')
pkgs = ['transformers>=4.40.0','accelerate','bitsandbytes',
        'datasets','scikit-learn','wandb','rich']
for p in pkgs:
    subprocess.run(['pip','install','-q',p], capture_output=True)
    print(f'  ✅ {p.split(">")[0]}')

import torch
print(f'\n✅ PyTorch: {torch.__version__}')
print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
if torch.cuda.is_available():
    print(f'✅ VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')

In [ ]:
# ── Cell 2: Clone Repo ────────────────────────────────────────────────────
import os, sys

GITHUB_REPO = 'https://github.com/vishalgwu/Latent-Instability-Diagnostics-LID.git'
REPO_DIR    = '/content/Latent-Instability-Diagnostics-LID'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
    print('✅ Repo updated')
else:
    !git clone {GITHUB_REPO} {REPO_DIR}
    print('✅ Repo cloned')

%cd {REPO_DIR}
!pip install -e .
sys.path.insert(0, REPO_DIR)
print('✅ LID package installed and ready')

In [ ]:
# ── Cell 3: Load Model ────────────────────────────────────────────────────
# FIX: load_in_8bit=True  — Llama-3-8B fp16 needs ~16 GB; GPU is 14.56 GB.
# 8-bit quantisation cuts the model footprint to ~9 GB, leaving ~5 GB free
# for activations, KV-cache, and detector overhead.
# NOTE: do NOT pass torch_dtype= when using bitsandbytes quantisation —
# the two flags conflict and bitsandbytes will raise a warning or error.

import os, time, torch
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

MODEL_ID = "meta-llama/Meta-Llama-3-8B"
HF_TOKEN = "PASTE_YOUR_HF_TOKEN_HERE"   # ← replace

login(token=HF_TOKEN)

print(f"Loading {MODEL_ID} in 8-bit ...")
t0 = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
torch.cuda.empty_cache()

print(f"✅ Model loaded in {time.time()-t0:.1f}s")
print(f"✅ VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"✅ VRAM free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated())/1e9:.2f} GB")

In [ ]:
# ── Cell 4: Load TruthfulQA Dev Set ──────────────────────────────────────
from datasets import load_dataset

print('Loading TruthfulQA...')
dataset = load_dataset('truthful_qa', 'generation', split='validation')

N_EXAMPLES = 100

examples = []
for ex in dataset:
    q = ex.get("question", "").strip()
    a = ex.get("best_answer", "").strip()
    if q and a and len(a.split()) > 2:
        examples.append(ex)
    if len(examples) >= N_EXAMPLES:
        break

print(f'✅ TruthfulQA loaded: {len(examples)} clean examples')
print(f'\nExample 0:')
print(f'Q: {examples[0]["question"]}')
print(f'A: {examples[0]["best_answer"]}')

In [ ]:
# ── Cell 5: Annotation Helper ─────────────────────────────────────────────
# Token is "hallucinated" if it does NOT appear in the gold answer tokens.
# This is a PROXY — full human annotation is Week 3.
# Using same annotation for all 3 baselines → fair relative comparison.

import numpy as np

def auto_annotate_tokens(generated_tokens, gold_answer_tokens):
    """
    label=0  token appears in gold answer  (likely correct)
    label=1  token does NOT appear         (proxy hallucination)
    """
    gold_set = set(gold_answer_tokens)
    labels = []
    for tok in generated_tokens:
        tok_clean = tok.strip().lower()
        is_hallucinated = tok_clean not in gold_set and len(tok_clean) > 2
        labels.append(1 if is_hallucinated else 0)
    return np.array(labels, dtype=float)

def tokenize_answer(text, tokenizer):
    """Return set of decoded token strings from answer text."""
    ids = tokenizer.encode(text.lower())
    return set(tokenizer.decode([i]).strip() for i in ids)

print('✅ Annotation helper ready')
print('   Note: automated token annotation (proxy for human labels)')
print('   Full human annotation pipeline → Week 3')

In [ ]:
"""
baselines/dola/detector.py
==========================
DoLA — Decoding by Contrasting Layers
Paper : Chuang et al., ICLR 2024
Link  : https://arxiv.org/abs/2309.03883

Core Idea:
    For each generated token, compute the Jensen-Shannon Divergence (JSD)
    between the vocabulary distribution from a "premature" (early) layer
    and the final layer.

    Score per token:
        score(t) = JSD( p_premature(t) || p_final(t) )

    Where:
        p_final(t)     = softmax(logits from last layer at token t)
        p_premature(t) = softmax(logits from early layer L_pre at token t)
        JSD            = symmetric KL divergence, range [0, 1]

    Higher score = more disagreement between layers = more likely hallucination

Target reproduction numbers (Chuang et al., TruthfulQA, Llama-7B):
    AUROC ≈ 0.65–0.68  (we need to be within ±2pp)

Implementation notes:
    - We use the layer-wise logit projection via the model's lm_head
    - Premature layer = layer at 40% depth (tunable via config)
    - No per-model fine-tuning required

NaN Root Cause & Fix (documented for reproducibility):
    Intermediate hidden states at premature_layer are captured BEFORE the
    final LayerNorm normalises them.  Projecting them through lm_head
    (calibrated for post-norm representations) produces logits that can
    reach ±1e5+ in fp16/bf16.

    torch.nan_to_num() with DEFAULT posinf converts inf → 3.4e38, but
    exp(3.4e38) overflows back to inf inside softmax → NaN probabilities.

    Fix applied in four layers:
      1. Hook casts to float32 immediately (prevents fp16 inf accumulation)
      2. model.model.norm + model.lm_head temporarily promoted to float32
         for the premature projection, then restored to original dtype
      3. nan_to_num with EXPLICIT posinf=50 / neginf=-50 on logits
         (NOT the default 3.4e38 which still overflows in exp)
      4. Hard clamp to [-50, 50] before softmax
         (exp(50) ≈ 5e21 — well within float32 range)
      5. nan_to_num after softmax as final safety net (uniform fallback)

Author : MIT LID Research Team
Week   : 2 — Baseline Implementation
"""

from __future__ import annotations

import time
import torch
import torch.nn.functional as F
from typing import Optional

import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(__file__))))

from baselines.base import BaseDetector, DetectorConfig, DetectorOutput
from dataclasses import dataclass as _dataclass


@_dataclass
class DoLAConfig(DetectorConfig):
    """DoLA-specific configuration."""
    premature_layer_ratio: float = 0.4
    premature_layer_idx: Optional[int] = None
    temperature: float = 1.0


class DoLADetector(BaseDetector):
    """
    DoLA hallucination detector.

    Scores each token by measuring how much the vocabulary distribution
    changes between an early (premature) layer and the final layer.
    High change = high uncertainty = likely hallucination.

    Output contract: scores [B, T], higher = more likely hallucinated.
    """

    def __init__(self, config: Optional[DoLAConfig] = None):
        if config is None:
            config = DoLAConfig(name="dola")
        super().__init__(config)
        self.dola_config = config

    @property
    def name(self) -> str:
        return "dola"

    # ──────────────────────────────────────────────────────────────────────
    # Private helpers
    # ──────────────────────────────────────────────────────────────────────

    def _get_premature_layer(self, n_layers: int) -> int:
        """Return premature layer index from config."""
        if self.dola_config.premature_layer_idx is not None:
            return self.dola_config.premature_layer_idx
        return int(n_layers * self.dola_config.premature_layer_ratio)

    def _safe_softmax(
        self,
        logits: torch.Tensor,
        temperature: float = 1.0,
    ) -> torch.Tensor:
        """
        Logits → probabilities with full overflow protection.

        Steps:
          1. Cast to float32 (no-op if already fp32)
          2. nan_to_num with EXPLICIT bounds — default posinf=3.4e38 still
             overflows inside exp(); capping at 50 keeps exp(50) ≈ 5e21
          3. Hard clamp (redundant safety; catches any edge case from step 2)
          4. F.softmax (internally uses x-max normalisation, so no overflow)
          5. nan_to_num after softmax — if an entire row was all-zero input
             (degenerate hidden state), the output would be NaN; replace with
             uniform distribution rather than propagating NaN downstream
        """
        logits = logits.float()
        logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)
        logits = logits.clamp(-50.0, 50.0)
        probs  = F.softmax(logits / temperature, dim=-1)
        vocab  = probs.shape[-1]
        probs  = torch.nan_to_num(probs, nan=1.0 / vocab)
        return probs

    def _jsd(
        self,
        p: torch.Tensor,
        q: torch.Tensor,
        eps: float = 1e-8,
    ) -> torch.Tensor:
        """
        Numerically stable Jensen-Shannon Divergence.

        JSD(p||q) = 0.5 * KL(p||m) + 0.5 * KL(q||m),  m = 0.5*(p+q)
        Range: [0, ln2] ≈ [0, 0.693]

        Stability measures applied in order:
          1. nan_to_num on inputs — stops NaN probs from propagating
          2. clamp(min=eps) — prevents log(0) = -inf
          3. renormalize — clamp shifts probability mass; dividing by sum
             restores the valid probability simplex before the KL computation
          4. nan_to_num on output — final catch-all

        Args:
            p, q : Probability distributions [..., vocab_size], float32
            eps  : Floor for probability values before log

        Returns:
            JSD scores [...]
        """
        # 1. Sanitise inputs
        p = torch.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
        q = torch.nan_to_num(q, nan=0.0, posinf=0.0, neginf=0.0)

        # 2. Floor (must happen before renorm, not after)
        p = p.clamp(min=eps)
        q = q.clamp(min=eps)

        # 3. Renormalize — clamp shifts mass, KL formula requires sum == 1
        p = p / p.sum(dim=-1, keepdim=True)
        q = q / q.sum(dim=-1, keepdim=True)

        # 4. Mixture distribution
        m = (0.5 * (p + q)).clamp(min=eps)

        # 5. KL divergences in float32 (both operands already fp32)
        kl_pm = (p * (p.log() - m.log())).sum(dim=-1)
        kl_qm = (q * (q.log() - m.log())).sum(dim=-1)

        jsd = 0.5 * (kl_pm + kl_qm)

        # 6. Final safety net
        return torch.nan_to_num(jsd, nan=0.0, posinf=0.0, neginf=0.0)

    # ──────────────────────────────────────────────────────────────────────
    # Public API
    # ──────────────────────────────────────────────────────────────────────

    def score(
        self,
        model,
        tokenizer,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> DetectorOutput:
        """
        Score each token using DoLA layer-contrast method.

        For each token position t:
            1. Capture hidden state at premature layer L_pre via hook
            2. Project through lm_head in float32 → premature logits
            3. Get final layer logits (standard forward pass)
            4. score(t) = JSD(softmax(premature), softmax(final))

        Args:
            model          : HuggingFace CausalLM (Llama / Mistral / etc.)
            tokenizer      : HuggingFace tokenizer
            input_ids      : [B, T] input token IDs
            attention_mask : optional [B, T]

        Returns:
            DetectorOutput with scores [B, T]
        """
        device    = next(model.parameters()).device
        input_ids = input_ids.to(device)

        n_layers    = model.config.num_hidden_layers
        pre_layer   = self._get_premature_layer(n_layers)

        # ── Hook: capture premature hidden state in float32 ────────────────
        captured = {}

        def hook_fn(module, input, output):
            h = output[0] if isinstance(output, tuple) else output
            # Cast to float32 immediately — fp16 inf must NOT enter the dict
            captured["h"] = h.detach().float()

        hook = model.model.layers[pre_layer].register_forward_hook(hook_fn)

        t0 = time.perf_counter()
        try:
            with torch.no_grad():
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
        finally:
            hook.remove()
        inference_time = time.perf_counter() - t0

        # ── Final-layer probabilities ──────────────────────────────────────
        final_probs = self._safe_softmax(
            outputs.logits, self.dola_config.temperature
        )

        # ── Premature-layer probabilities ──────────────────────────────────
        # Clean the captured hidden state
        h = torch.nan_to_num(
            captured["h"], nan=0.0, posinf=1e4, neginf=-1e4
        )

        # Apply final LayerNorm in float32.
        #
        # IMPORTANT — we must NEVER call model.model.norm.float() or
        # model.lm_head.float() to promote the module in-place.
        # If an OOM (or any exception) fires mid-computation, the restore
        # never runs and the model is permanently left in the wrong dtype,
        # causing c10::Half != float on every subsequent forward pass.
        #
        # Safe pattern: extract .weight / .bias as float32 TENSORS
        # (this creates a new tensor — the module's stored parameter is
        # untouched) and compute with F.layer_norm / F.linear directly.
        with torch.no_grad():
            try:
                norm_w = model.model.norm.weight.float()          # new fp32 tensor
                norm_b = getattr(model.model.norm, "bias", None)
                norm_b = norm_b.float() if norm_b is not None else None
                eps    = getattr(model.model.norm, "variance_epsilon",
                                 getattr(model.model.norm, "eps", 1e-5))
                # Llama / Mistral use RMSNorm (no bias, no mean subtraction)
                variance = h.pow(2).mean(-1, keepdim=True)
                h_normed = h * torch.rsqrt(variance + eps) * norm_w
                if norm_b is not None:
                    h_normed = h_normed + norm_b
            except Exception:
                # Fallback for unknown norm architectures: use raw hidden state
                h_normed = h

        # Project through lm_head using F.linear with a float32 copy of
        # the weight — no module state is mutated at any point.
        with torch.no_grad():
            lm_w      = model.lm_head.weight.float()              # new fp32 tensor
            lm_b      = model.lm_head.bias
            lm_b      = lm_b.float() if lm_b is not None else None
            pre_logits = F.linear(h_normed, lm_w, lm_b)          # [B, T, vocab]

        pre_probs = self._safe_softmax(
            pre_logits, self.dola_config.temperature
        )

        # ── JSD score per token ────────────────────────────────────────────
        scores = self._jsd(pre_probs, final_probs)   # [B, T]

        # NaN audit: after all guards there should be zero NaN.
        # If any survive, warn loudly (useful for debugging new model archs).
        n_nan = scores.isnan().sum().item()
        if n_nan > 0:
            print(
                f"[DoLA WARNING] {n_nan} NaN scores survived all sanitisation "
                f"layers.  Check: (1) model loaded with torch_dtype= not dtype=, "
                f"(2) input_ids contains no out-of-range token IDs."
            )
            scores = torch.nan_to_num(scores, nan=0.0)

        # ── Decode tokens ──────────────────────────────────────────────────
        tokens = [
            [tokenizer.decode([tok_id]) for tok_id in seq]
            for seq in input_ids.cpu().tolist()
        ]

        return DetectorOutput(
            scores=scores.cpu(),
            tokens=tokens,
            metadata={
                "premature_layer":       pre_layer,
                "n_layers":              n_layers,
                "premature_layer_ratio": self.dola_config.premature_layer_ratio,
                "inference_time_s":      inference_time,
            },
        )

    def score_generated(
        self,
        model,
        tokenizer,
        prompt: str,
        max_new_tokens: int = 200,
    ) -> dict:
        """
        Generate a response and score each generated token.

        Returns:
            generated_text : str
            token_scores   : list[float]  — one score per generated token
            tokens         : list[str]    — decoded generated tokens
            prompt_len     : int          — number of prompt tokens (for slicing)
        """
        device = next(model.parameters()).device
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        result     = self.score(model, tokenizer, gen_ids)
        prompt_len = inputs["input_ids"].shape[1]

        return {
            "generated_text": tokenizer.decode(
                gen_ids[0, prompt_len:], skip_special_tokens=True
            ),
            "token_scores":  result.scores[0, prompt_len:].tolist(),
            "tokens":        result.tokens[0][prompt_len:],
            "prompt_len":    prompt_len,
        }


In [ ]:
"""
baselines/ssp/detector.py
=========================
SSP — Semantic Self-consistency via input Perturbation
Paper : Luo et al., June 2025
Status: CLOSEST competitor to LID

Core Idea:
    Perturbs the INPUT (token embeddings) and measures how much the
    output probability distribution changes.  High sensitivity to input
    perturbation = unstable prediction = likely hallucination.

    CRITICAL DIFFERENCE from LID:
        SSP  : perturbs INPUT embeddings  → measures output distribution shift
        LID  : perturbs HIDDEN STATES     → measures internal representation shift

    LID advantage:
        - More targeted: perturbation at the exact layer where instability occurs
        - Computes I(t) AND q(t) — SSP only gets magnitude shift
        - Can detect instability token-by-token during generation
        - SSP needs a full second forward pass; LID uses a partial pass

    Score per token t:
        SSP_score(t) = JSD( p_clean(t) || p_perturbed(t) )

    Where:
        p_clean(t)     = output prob distribution with original embeddings
        p_perturbed(t) = output prob distribution with noisy embeddings
        JSD            = Jensen-Shannon Divergence ∈ [0, 1]

    Higher score = more sensitive to input noise = likely hallucination

Target reproduction numbers (Luo 2025):
    AUROC ≈ 0.64–0.68 on TruthfulQA
    NOTE: Paper has ambiguous hyperparameters — all assumptions documented below

Hyperparameter assumptions (document in fidelity report):
    - alpha = 0.05  (5% of embedding RMS — same as LID default for fair comparison)
    - n_samples = 1 (single perturbation — paper is ambiguous on ensemble size)
    - Perturbation applied at embedding layer output (before first transformer block)

Author : MIT LID Research Team
Week   : 2 — Baseline Implementation
"""

from __future__ import annotations

import torch
import torch.nn.functional as F
from typing import Optional

import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(__file__))))

from baselines.base import BaseDetector, DetectorConfig, DetectorOutput
from dataclasses import dataclass as _dataclass


@_dataclass
class SSPConfig(DetectorConfig):
    """SSP-specific configuration."""
    alpha: float = 0.05        # noise magnitude relative to embedding RMS
    n_samples: int = 1         # number of perturbation samples (paper ambiguous)
    perturb_seed: int = 42     # base seed; sample i uses seed + i


class SSPDetector(BaseDetector):
    """
    SSP hallucination detector.

    Perturbs INPUT embeddings and measures how much the output
    probability distribution changes under that perturbation.

    This is the closest published method to LID:
    - Both use perturbation + distribution shift measurement
    - SSP operates on input space; LID operates on hidden state space
    - SSP is the primary comparison point for the 'perturbation' novelty claim

    Output contract: scores [B, T], higher = more likely hallucinated.
    """

    def __init__(self, config: Optional[SSPConfig] = None):
        if config is None:
            config = SSPConfig(name="ssp")
        super().__init__(config)
        self.ssp_config = config

    @property
    def name(self) -> str:
        return "ssp"

    # ──────────────────────────────────────────────────────────────────────
    # Private helpers
    # ──────────────────────────────────────────────────────────────────────

    def _rms(self, h: torch.Tensor) -> torch.Tensor:
        """
        Root-mean-square of h across the last dimension.
        RMS(h) = sqrt( mean(h²) )

        Returns shape [..., 1] for broadcasting with h.
        """
        return h.pow(2).mean(dim=-1, keepdim=True).sqrt()

    def _inject_noise(
        self,
        embeddings: torch.Tensor,
        alpha: float,
        seed: int,
    ) -> torch.Tensor:
        """
        Add scaled Gaussian noise to embeddings.

        δ ~ N(0, σ²I),   σ = alpha × RMS(embedding)
        h_pert = h + δ

        Deterministic: the same (seed, alpha, embeddings) always produces
        the same δ, enabling exact reproducibility across runs.

        Args:
            embeddings : float32 tensor [B, T, d_model]
            alpha      : noise scale relative to embedding magnitude
            seed       : RNG seed for reproducibility

        Returns:
            Perturbed embeddings, same shape and dtype as input
        """
        generator = torch.Generator(device=embeddings.device)
        generator.manual_seed(seed)
        sigma = alpha * self._rms(embeddings)            # [B, T, 1]
        delta = torch.randn_like(embeddings, generator=generator) * sigma
        return embeddings + delta

    def _jsd(
        self,
        p: torch.Tensor,
        q: torch.Tensor,
        eps: float = 1e-8,
    ) -> torch.Tensor:
        """
        Numerically stable Jensen-Shannon Divergence.

        JSD(p||q) = 0.5 * KL(p||m) + 0.5 * KL(q||m),  m = 0.5*(p+q)
        Range: [0, ln2] ≈ [0, 0.693]

        Args:
            p, q : Probability distributions [..., vocab_size], float32
            eps  : Floor for probability values before log

        Returns:
            JSD scores [...]
        """
        # Sanitise inputs
        p = torch.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
        q = torch.nan_to_num(q, nan=0.0, posinf=0.0, neginf=0.0)

        # Floor → renormalize → mixture
        p = p.clamp(min=eps)
        q = q.clamp(min=eps)
        p = p / p.sum(dim=-1, keepdim=True)
        q = q / q.sum(dim=-1, keepdim=True)
        m = (0.5 * (p + q)).clamp(min=eps)

        kl_pm = (p * (p.log() - m.log())).sum(dim=-1)
        kl_qm = (q * (q.log() - m.log())).sum(dim=-1)

        jsd = 0.5 * kl_pm + 0.5 * kl_qm
        return torch.nan_to_num(jsd, nan=0.0, posinf=0.0, neginf=0.0)

    def _logits_to_probs(self, logits: torch.Tensor) -> torch.Tensor:
        """
        Safe logits → probabilities.
        Handles overflow the same way as DoLA for consistent treatment.
        """
        logits = logits.float()
        logits = torch.nan_to_num(logits, nan=0.0, posinf=50.0, neginf=-50.0)
        logits = logits.clamp(-50.0, 50.0)
        probs  = F.softmax(logits, dim=-1)
        vocab  = probs.shape[-1]
        return torch.nan_to_num(probs, nan=1.0 / vocab)

    # ──────────────────────────────────────────────────────────────────────
    # Public API
    # ──────────────────────────────────────────────────────────────────────

    def score(
        self,
        model,
        tokenizer,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> DetectorOutput:
        """
        Score each token using SSP input-perturbation method.

        For each token position t:
            1. Run clean forward pass → p_clean(t)
            2. Perturb input embeddings with Gaussian noise
            3. Run perturbed forward pass → p_perturbed(t)
            4. score(t) = JSD(p_clean(t), p_perturbed(t))
            (Optionally averaged over n_samples perturbations)

        Args:
            model          : HuggingFace CausalLM
            tokenizer      : HuggingFace tokenizer
            input_ids      : [B, T]
            attention_mask : optional [B, T]

        Returns:
            DetectorOutput with scores [B, T]
        """
        device    = next(model.parameters()).device
        input_ids = input_ids.to(device)
        model_dtype = next(model.parameters()).dtype

        # ── Clean forward pass ────────────────────────────────────────────
        with torch.no_grad():
            # Get embeddings in float32 for noise injection
            clean_embeds_fp32 = model.model.embed_tokens(input_ids).float()

            clean_outputs = model(
                inputs_embeds=clean_embeds_fp32.to(model_dtype),
                attention_mask=attention_mask,
            )
            clean_probs = self._logits_to_probs(clean_outputs.logits)

        # ── Perturbed forward pass(es) ────────────────────────────────────
        pert_probs_list = []

        for sample_idx in range(self.ssp_config.n_samples):
            seed = self.ssp_config.perturb_seed + sample_idx

            with torch.no_grad():
                pert_embeds_fp32 = self._inject_noise(
                    clean_embeds_fp32,
                    alpha=self.ssp_config.alpha,
                    seed=seed,
                )
                pert_outputs = model(
                    inputs_embeds=pert_embeds_fp32.to(model_dtype),
                    attention_mask=attention_mask,
                )
                pert_probs_list.append(
                    self._logits_to_probs(pert_outputs.logits)
                )

        # Average over samples if n_samples > 1
        avg_pert_probs = torch.stack(pert_probs_list, dim=0).mean(dim=0)

        # ── JSD score per token ───────────────────────────────────────────
        scores = self._jsd(clean_probs, avg_pert_probs)   # [B, T]

        # ── Decode tokens ─────────────────────────────────────────────────
        tokens = [
            [tokenizer.decode([tok_id]) for tok_id in seq]
            for seq in input_ids.cpu().tolist()
        ]

        return DetectorOutput(
            scores=scores.cpu(),
            tokens=tokens,
            metadata={
                "alpha":              self.ssp_config.alpha,
                "n_samples":          self.ssp_config.n_samples,
                "perturbation_level": "input_embeddings",
                "assumption":         (
                    "alpha=0.05, single perturbation "
                    "(Luo 2025 hyperparameters ambiguous — see fidelity report)"
                ),
            },
        )

    def score_generated(
        self,
        model,
        tokenizer,
        prompt: str,
        max_new_tokens: int = 200,
    ) -> dict:
        """
        Generate response and score each generated token.

        Returns:
            generated_text : str
            token_scores   : list[float]  — one score per generated token
            tokens         : list[str]    — decoded generated tokens
            prompt_len     : int          — number of prompt tokens (for slicing)
        """
        device = next(model.parameters()).device
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        result     = self.score(model, tokenizer, gen_ids)
        prompt_len = inputs["input_ids"].shape[1]

        return {
            "generated_text": tokenizer.decode(
                gen_ids[0, prompt_len:], skip_special_tokens=True
            ),
            "token_scores":  result.scores[0, prompt_len:].tolist(),
            "tokens":        result.tokens[0][prompt_len:],
            "prompt_len":    prompt_len,
        }


In [ ]:
"""
baselines/lsd/detector.py
=========================
LSD — Layer-wise Semantic Drift
Paper : arXiv:2510.04933 (2024)
Status: MOST SIMILAR prior work to LID — we MUST beat it on lead time

Core Idea:
    Measures how much the semantic representation of a token DRIFTS
    as it passes through successive transformer layers.  High drift
    between layers = model hasn't settled on a stable representation
    = uncertain = likely to hallucinate.

    Unlike LID which perturbs hidden states, LSD measures natural
    layer-to-layer variation without any perturbation.

    Score per token t at position p:
        drift(l, t) = 1 - cosine( h_l(t), h_{l+1}(t) )
        LSD_score(t) = aggregate( drift(l, t) for l in selected_layers )

    Higher score = more semantic instability = more likely hallucination

Key Difference from LID:
    LSD  : measures drift between CONSECUTIVE layers (natural variation)
    LID  : measures response to PERTURBATION        (controlled sensitivity test)

    LID advantage: LID's perturbation isolates sensitivity from normal
    representation evolution — more targeted signal.

Target reproduction numbers (arXiv:2510.04933):
    AUROC ≈ 0.63–0.67 on TruthfulQA with Llama-7B

Author : MIT LID Research Team
Week   : 2 — Baseline Implementation
"""

from __future__ import annotations

import torch
import torch.nn.functional as F
from typing import Optional, List, Tuple

import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.dirname(__file__))))

from baselines.base import BaseDetector, DetectorConfig, DetectorOutput
from dataclasses import dataclass as _dataclass, field as _field


@_dataclass
class LSDConfig(DetectorConfig):
    """LSD-specific configuration."""
    # List of (l1, l2) pairs to measure drift between.
    # None = all consecutive pairs (l, l+1) for l in range(n_layers - 1)
    layer_pairs: Optional[List[Tuple[int, int]]] = None

    # Whether to apply depth weighting in weighted_mean aggregation
    use_depth_weighting: bool = True

    # Aggregation strategy: "mean" | "max" | "weighted_mean"
    aggregation: str = "mean"


class LSDDetector(BaseDetector):
    """
    LSD hallucination detector.

    Measures semantic drift between consecutive transformer layers.
    Tokens with high layer-to-layer drift are flagged as likely hallucinations.

    Key insight: before hallucination, the model's internal representation
    keeps changing across layers (hasn't converged to a stable answer).
    For factual tokens, representations stabilize in middle-to-late layers.

    Output contract: scores [B, T], higher = more likely hallucinated.
    """

    def __init__(self, config: Optional[LSDConfig] = None):
        if config is None:
            config = LSDConfig(name="lsd")
        super().__init__(config)
        self.lsd_config = config

    @property
    def name(self) -> str:
        return "lsd"

    # ──────────────────────────────────────────────────────────────────────
    # Private helpers
    # ──────────────────────────────────────────────────────────────────────

    def _cosine_drift(
        self,
        h1: torch.Tensor,
        h2: torch.Tensor,
        eps: float = 1e-8,
    ) -> torch.Tensor:
        """
        Compute semantic drift between two hidden states.

        drift = 1 - cosine_similarity(h1, h2)

        Range: [0, 2]
            0   = no drift (identical representations)
            1   = orthogonal (complete drift)
            2   = antiparallel (reversed direction)

        Uses F.normalize with eps guard to prevent division by zero when
        a hidden state is the zero vector (can happen with bad inputs or
        degenerate activations).

        Args:
            h1, h2 : Hidden states [B, T, d_model], float32

        Returns:
            drift  : [B, T]
        """
        h1_norm = F.normalize(h1, p=2, dim=-1, eps=eps)
        h2_norm = F.normalize(h2, p=2, dim=-1, eps=eps)
        cosine_sim = (h1_norm * h2_norm).sum(dim=-1)   # [B, T]
        drift = 1.0 - cosine_sim
        return torch.nan_to_num(drift, nan=0.0, posinf=0.0, neginf=0.0)

    # ──────────────────────────────────────────────────────────────────────
    # Public API
    # ──────────────────────────────────────────────────────────────────────

    def score(
        self,
        model,
        tokenizer,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> DetectorOutput:
        """
        Score each token using LSD layer-drift method.

        For each token position t:
            1. Capture hidden states at ALL layers via hooks
            2. Compute drift(l, l+1) for each consecutive layer pair
            3. Aggregate drift scores across layers
            4. score(t) = aggregated drift

        Args:
            model          : HuggingFace CausalLM
            tokenizer      : HuggingFace tokenizer
            input_ids      : [B, T]
            attention_mask : optional [B, T]

        Returns:
            DetectorOutput with scores [B, T]
        """
        device    = next(model.parameters()).device
        input_ids = input_ids.to(device)
        n_layers  = model.config.num_hidden_layers

        # ── Register hooks on ALL layers ──────────────────────────────────
        all_hidden: dict = {}
        hooks = []

        for layer_idx in range(n_layers):
            def make_hook(idx: int):
                def hook(module, inp, output):
                    h = output[0] if isinstance(output, tuple) else output
                    # Sanitise and cast to float32 in hook to avoid fp16 inf
                    all_hidden[idx] = torch.nan_to_num(
                        h.detach().float(),
                        nan=0.0, posinf=1e4, neginf=-1e4,
                    )
                return hook
            hooks.append(
                model.model.layers[layer_idx].register_forward_hook(
                    make_hook(layer_idx)
                )
            )

        try:
            with torch.no_grad():
                _ = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
        finally:
            for h in hooks:
                h.remove()

        # ── Determine which layer pairs to use ────────────────────────────
        if self.lsd_config.layer_pairs is not None:
            pairs = self.lsd_config.layer_pairs
        else:
            pairs = [(l, l + 1) for l in range(n_layers - 1)]

        # ── Compute per-pair cosine drift ─────────────────────────────────
        drift_per_pair = []
        for l1, l2 in pairs:
            h1 = all_hidden[l1]   # [B, T, d_model], float32
            h2 = all_hidden[l2]
            drift_per_pair.append(self._cosine_drift(h1, h2))   # [B, T]

        drift_stack = torch.stack(drift_per_pair, dim=0)   # [n_pairs, B, T]

        # ── Aggregate across layer pairs ──────────────────────────────────
        agg = self.lsd_config.aggregation

        if agg == "max":
            scores = drift_stack.max(dim=0).values

        elif agg == "weighted_mean" and self.lsd_config.use_depth_weighting:
            # Linear depth weighting: deeper layer pairs weighted higher
            # Rationale: later layers carry more semantically meaningful signal
            n_p     = len(pairs)
            weights = torch.linspace(
                1.0, float(n_p), n_p, device=drift_stack.device
            )
            weights = weights / weights.sum()
            scores  = (drift_stack * weights[:, None, None]).sum(dim=0)

        else:
            # Default: simple mean across all layer pairs
            scores = drift_stack.mean(dim=0)

        scores = torch.nan_to_num(scores, nan=0.0)

        # ── Decode tokens ─────────────────────────────────────────────────
        tokens = [
            [tokenizer.decode([tok_id]) for tok_id in seq]
            for seq in input_ids.cpu().tolist()
        ]

        return DetectorOutput(
            scores=scores.cpu(),
            tokens=tokens,
            metadata={
                "n_layer_pairs":      len(pairs),
                "aggregation":        agg,
                "layer_range":        f"0-{n_layers - 1}",
                "drift_global_mean":  float(drift_stack.mean().item()),
            },
        )

    def score_generated(
        self,
        model,
        tokenizer,
        prompt: str,
        max_new_tokens: int = 200,
    ) -> dict:
        """
        Generate response and score each generated token.

        Returns:
            generated_text : str
            token_scores   : list[float]  — one score per generated token
            tokens         : list[str]    — decoded generated tokens
            prompt_len     : int          — number of prompt tokens (for slicing)
        """
        device = next(model.parameters()).device
        inputs = tokenizer(prompt, return_tensors="pt").to(device)

        with torch.no_grad():
            gen_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        result     = self.score(model, tokenizer, gen_ids)
        prompt_len = inputs["input_ids"].shape[1]

        return {
            "generated_text": tokenizer.decode(
                gen_ids[0, prompt_len:], skip_special_tokens=True
            ),
            "token_scores":  result.scores[0, prompt_len:].tolist(),
            "tokens":        result.tokens[0][prompt_len:],
            "prompt_len":    prompt_len,
        }


In [ ]:
# ── Cell 9: Run All Baselines + Compute Metrics ───────────────────────────

from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
import numpy as np
from dataclasses import dataclass
from typing import Optional, List
import time, warnings, torch

warnings.filterwarnings("ignore", category=RuntimeWarning)


@dataclass
class BaselineResult:
    auroc:              Optional[float]       = None
    auroc_ci:           Optional[List[float]] = None
    auprc:              Optional[float]       = None
    fpr_at_tpr80:       Optional[float]       = None
    hallucination_rate: Optional[float]       = None
    n_tokens:           Optional[int]         = None
    below_threshold:    bool                  = True


def bootstrap_auroc_ci(y_true, y_score, n_boot=300, ci=0.95):
    rng = np.random.RandomState(42)
    aurocs, n = [], len(y_true)
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        yt, ys = y_true[idx], y_score[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            aurocs.append(roc_auc_score(yt, ys))
        except Exception:
            pass
    if len(aurocs) < 10:
        return [float("nan"), float("nan")]
    alpha = (1 - ci) / 2
    return [float(np.percentile(aurocs, 100*alpha)),
            float(np.percentile(aurocs, 100*(1-alpha)))]


def fpr_at_tpr(y_true, y_score, target_tpr=0.80):
    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_score)
    idx = np.searchsorted(tpr_arr, target_tpr)
    return float(fpr_arr[min(idx, len(fpr_arr)-1)])


def run_detector_on_examples(detector, examples, tokenizer, max_new_tokens=60):
    all_scores, all_labels = [], []
    n_ok, n_oom, n_other = 0, 0, 0
    n = len(examples)

    for i, ex in enumerate(examples):
        print(f"  [{i+1:3d}/{n}]  ok={n_ok} oom={n_oom} err={n_other}", end="\r")

        # Free KV-cache and activations from previous example before next run
        torch.cuda.empty_cache()

        q      = ex["question"].strip()
        gold   = ex["best_answer"].strip()
        prompt = f"Q: {q}\nA:"

        try:
            out = detector.score_generated(
                model, tokenizer, prompt, max_new_tokens=max_new_tokens
            )
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            n_oom += 1
            print(f"\n  ⚠️  [{i}] OOM — skipping. If this repeats, reduce max_new_tokens further.")
            continue
        except Exception as e:
            n_other += 1
            print(f"\n  ⚠️  [{i}] {type(e).__name__}: {e}")
            continue

        raw_scores = np.array(out["token_scores"], dtype=float)
        tokens     = out["tokens"]

        if raw_scores.size == 0 or np.all(np.isnan(raw_scores)):
            n_other += 1
            continue

        raw_scores = np.nan_to_num(raw_scores, nan=0.0)
        gold_toks  = tokenize_answer(gold, tokenizer)
        raw_labels = auto_annotate_tokens(tokens, gold_toks)

        min_len = min(len(raw_scores), len(raw_labels))
        all_scores.extend(raw_scores[:min_len].tolist())
        all_labels.extend(raw_labels[:min_len].tolist())
        n_ok += 1

    print(f"\n  ✅ Done — ok={n_ok}  oom={n_oom}  other={n_other}  "
          f"tokens={len(all_scores):,}")
    return np.array(all_scores, dtype=float), np.array(all_labels, dtype=float)


# ── Instantiate detectors ─────────────────────────────────────────────────
detectors = {
    "dola": DoLADetector(),
    "ssp":  SSPDetector(),
    "lsd":  LSDDetector(),
}

THRESHOLDS = {"dola": 0.60, "lsd": 0.58, "ssp": 0.58}
results    = {}

for det_name, detector in detectors.items():
    print(f"\n{'='*54}")
    print(f"  Running {det_name.upper()} ...")
    print(f"{'='*54}")
    t0 = time.time()

    scores, labels = run_detector_on_examples(
        detector, examples, tokenizer, max_new_tokens=60
    )
    elapsed = time.time() - t0

    if len(scores) == 0 or len(np.unique(labels)) < 2:
        print(f"  ⚠️  Insufficient data for metrics (need ≥2 label classes)")
        results[det_name] = BaselineResult(
            hallucination_rate=float(labels.mean()) if len(labels) else None,
            n_tokens=int(len(scores)),
            below_threshold=True,
            auroc_ci=[float("nan"), float("nan")],
        )
        continue

    auroc  = float(roc_auc_score(labels, scores))
    auprc  = float(average_precision_score(labels, scores))
    ci     = bootstrap_auroc_ci(labels, scores)
    fpr80  = fpr_at_tpr(labels, scores, target_tpr=0.80)
    h_rate = float(labels.mean())
    threshold = THRESHOLDS.get(det_name, 0.58)
    below     = auroc < threshold

    results[det_name] = BaselineResult(
        auroc=auroc, auroc_ci=ci, auprc=auprc, fpr_at_tpr80=fpr80,
        hallucination_rate=h_rate, n_tokens=int(len(scores)),
        below_threshold=below,
    )

    status = "✅ PASS" if not below else f"⚠️  BELOW TARGET (≥{threshold:.2f})"
    print(f"  AUROC     : {auroc:.4f}  {status}")
    print(f"  AUPRC     : {auprc:.4f}")
    print(f"  FPR@TPR80 : {fpr80:.4f}")
    print(f"  Hall.rate : {h_rate:.2%}")
    print(f"  N tokens  : {len(scores):,}")
    print(f"  Time      : {elapsed:.1f}s")

print("\n✅ All baselines complete.  'results' dict is ready.")

In [ ]:
# ── Cell 10: Results Table ───────────────────────────────────────────────
import pandas as pd
import numpy as np

rows = []
for name, r in results.items():

    def safe_float(x, fmt="{:.3f}"):
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "N/A"
        return fmt.format(x)

    def safe_ci(ci):
        if ci is None or len(ci) != 2:
            return "N/A"
        if any(v is None or np.isnan(v) for v in ci):
            return "N/A"
        return f"[{ci[0]:.3f}, {ci[1]:.3f}]"

    rows.append({
        "Method":     name.upper(),
        "AUROC":      safe_float(r.auroc),
        "AUROC 95%CI": safe_ci(r.auroc_ci),
        "AUPRC":      safe_float(r.auprc),
        "FPR@TPR80":  safe_float(r.fpr_at_tpr80),
        "Hall.Rate":  safe_float(r.hallucination_rate, "{:.2%}"),
        "N tokens":   r.n_tokens if r.n_tokens is not None else 0,
        "RedFlag":    "⚠️" if getattr(r, "below_threshold", False) else "✅",
    })

df = pd.DataFrame(rows)

print("\n" + "="*70)
print("  WEEK 2 BASELINE RESULTS — TruthfulQA-dev-100, Llama-3-8B")
print("="*70)
print(df.to_string(index=False))
print("="*70)

print("\nTarget thresholds (from execution plan):")
print("  DoLA : AUROC ≥ 0.60 (within ±2pp of Chuang et al.)")
print("  LSD  : AUROC ≥ 0.58 (within ±2pp of arXiv:2510.04933)")
print("  SSP  : AUROC ≥ 0.58 (within ±3pp of Luo 2025)")
print("\nThese numbers are the bar LID must clear in Week 5.")

In [ ]:
# ── Cell 11: Save Results ─────────────────────────────────────────────────
import json, os
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

RESEARCH_DIR = '/content/drive/MyDrive/LID-Research'
os.makedirs(RESEARCH_DIR, exist_ok=True)

save_data = {
    'week': 2,
    'model': MODEL_ID,
    'dataset': 'TruthfulQA-dev-100',
    'annotation': 'automated_token_proxy',
    'results': {
        name: {
            'auroc':             r.auroc,
            'auprc':             r.auprc,
            'fpr_at_tpr80':      r.fpr_at_tpr80,
            'auroc_ci':          list(r.auroc_ci) if r.auroc_ci else [None, None],
            'hallucination_rate': r.hallucination_rate,
            'n_tokens':          r.n_tokens,
            'below_threshold':   r.below_threshold,
        }
        for name, r in results.items()
    }
}

out_path = os.path.join(RESEARCH_DIR, 'week2_baseline_results.json')
with open(out_path, 'w') as f:
    json.dump(save_data, f, indent=2)

df.to_csv(os.path.join(RESEARCH_DIR, 'week2_table.csv'), index=False)

print(f'✅ Results saved to: {out_path}')
print()
print('PASTE the week2_baseline_results.json content into your')
print('Friday advisor report (docs/weekly-reports/week02_report.md)')

In [ ]:
# ── Cell 12: Fidelity Check ───────────────────────────────────────────────
# Compare reproduced numbers against paper-reported numbers.
# Documents into docs/baseline-reproduction/*.md

PAPER_NUMBERS = {
    'dola': {'auroc': 0.665, 'source': 'Chuang et al. ICLR 2024, TruthfulQA, Llama-7B'},
    'lsd':  {'auroc': 0.640, 'source': 'arXiv:2510.04933, TruthfulQA'},
    'ssp':  {'auroc': 0.655, 'source': 'Luo 2025, TruthfulQA (approximate — preprint)'},
}

print('='*60)
print('  FIDELITY REPORT — Reproduction vs Paper Numbers')
print('='*60)
print(f'{"Method":<8} {"Ours":>8} {"Paper":>8} {"Delta":>8} {"Pass?":>8}')
print('-'*60)

for name, r in results.items():
    if name not in PAPER_NUMBERS:
        continue
    paper = PAPER_NUMBERS[name]
    ours  = r.auroc if r.auroc else 0.0
    delta = ours - paper['auroc']
    passed = abs(delta) <= 0.05   # ±5pp acceptable (Llama-3-8B vs Llama-2-7B)
    status = '✅ PASS' if passed else '⚠️  FAIL'
    sign   = '+' if delta >= 0 else ''
    print(f'{name.upper():<8} {ours:>8.3f} {paper["auroc"]:>8.3f} '
          f'{sign}{delta:>7.3f} {status:>8}')

print('-'*60)
print()
print('Note: Delta expected — Llama-3-8B vs Llama-2-7B architecture diff.')
print('Document ALL deviations in docs/baseline-reproduction/*.md')